In [1]:
!pip install -q sentence-transformers
import re, numpy as np, pandas as pd, torch
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sentence_transformers import SentenceTransformer
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

root = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
OUT  = "/kaggle/input/datasets/luvagnihotri0721/shl-transcripts-clean/transcripts.jsonl"
train = pd.read_csv(f"{root}/train.csv"); test = pd.read_csv(f"{root}/test.csv")
dev = "cuda" if torch.cuda.is_available() else "cpu"   # works with or without GPU
print("device:", dev)

device: cuda


In [2]:
# 1. Merge transcripts (key = split + filename)
tr = pd.read_json(OUT, lines=True).drop_duplicates(["split","filename"])
train_df = train.assign(split="train").merge(tr, on=["filename","split"], how="left")
test_df  = test.assign(split="test").merge(tr, on=["filename","split"], how="left")
for d in (train_df, test_df):
    d["text"] = d["text"].fillna(""); d["logprob"] = d["logprob"].fillna(-5.0)
    d["no_speech"] = d["no_speech"].fillna(1.0); d["duration"] = d["duration"].fillna(60.0)
print(len(train_df), len(test_df))   # expect 769 216

# 2. Hand-made text features
FILL = {"um","umm","uh","uhh","hmm","ah","ahh","like"}
def feats(r):
    t = r.text.lower(); w = re.findall(r"[a-z']+", t); n = max(len(w), 1)
    sents = [s for s in re.split(r"[.!?]+", t) if s.strip()]
    rep = sum(1 for a, b in zip(w, w[1:]) if a == b)
    fil = sum(1 for x in w if x in FILL)
    return dict(n_words=len(w), wps=len(w)/max(r.duration, 1), n_sents=len(sents),
                avg_sent=n/max(len(sents), 1), ttr=len(set(w))/n, rep_rate=rep/n,
                fill_rate=fil/n, avg_wlen=np.mean([len(x) for x in w]) if w else 0,
                logprob=r.logprob, no_speech=r.no_speech, duration=r.duration,
                ellipsis=t.count("..."))
Xtr = pd.DataFrame([feats(r) for r in train_df.itertuples()])
Xte = pd.DataFrame([feats(r) for r in test_df.itertuples()])
y = train_df.label.values; n = len(train_df)
texts = train_df.text.tolist() + test_df.text.tolist()

# 3. Sentence embeddings + GPT-2 fluency loss
E = SentenceTransformer("all-mpnet-base-v2", device=dev).encode(
        texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
tok = GPT2TokenizerFast.from_pretrained("gpt2")
lm = GPT2LMHeadModel.from_pretrained("gpt2").to(dev).eval()
def lm_loss(t):
    if len(t.split()) < 3: return 8.0
    ids = tok(t, return_tensors="pt", truncation=True, max_length=512).input_ids.to(dev)
    with torch.no_grad(): return lm(ids, labels=ids).loss.item()
H = pd.concat([Xtr, Xte], ignore_index=True)
H["lm_loss"] = [lm_loss(t) for t in texts]
Hs = StandardScaler().fit(H.iloc[:n]).transform(H)
XE = np.hstack([E, 0.15 * Hs])

# 4. Models, 5-fold CV, blend
models = {"ridge_H": (Hs, Ridge(alpha=10)),
          "ridge_EH": (XE, Ridge(alpha=3)),
          "svr_EH": (XE, SVR(C=3, epsilon=0.2))}
kf = KFold(5, shuffle=True, random_state=42)
oofs, trp, tep = {}, {}, {}
for k, (X, m) in models.items():
    o = np.zeros(n)
    for a, b in kf.split(X[:n]):
        m.fit(X[a], y[a]); o[b] = m.predict(X[b])
    oofs[k] = np.clip(o, 0, 5)
    m.fit(X[:n], y)
    trp[k] = np.clip(m.predict(X[:n]), 0, 5); tep[k] = np.clip(m.predict(X[n:]), 0, 5)
    print(k, "OOF RMSE %.4f Pearson %.4f" % (np.sqrt(mean_squared_error(y, oofs[k])), pearsonr(y, oofs[k])[0]))
bo, bt, bp = (np.mean(list(d.values()), 0) for d in (oofs, trp, tep))
print("BLEND OOF RMSE %.4f Pearson %.4f" % (np.sqrt(mean_squared_error(y, bo)), pearsonr(y, bo)[0]))
print("BLEND TRAIN RMSE %.4f" % np.sqrt(mean_squared_error(y, bt)))

# 5. Always write the submission
sub = pd.DataFrame({"filename": test_df.filename, "label": bp})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.shape, sub.label.isna().sum(), sub.label.min(), sub.label.max())

769 216


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


ridge_H OOF RMSE 0.9088 Pearson 0.6824
ridge_EH OOF RMSE 0.8802 Pearson 0.7119
svr_EH OOF RMSE 0.8596 Pearson 0.7204
BLEND OOF RMSE 0.8533 Pearson 0.7351
BLEND TRAIN RMSE 0.6406
(216, 2) 0 1.5761729572031822 4.887769698906978


In [3]:
import soundfile as sf
from scipy.signal import resample_poly
from transformers import AutoFeatureExtractor, AutoModel

fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wm = AutoModel.from_pretrained("microsoft/wavlm-base-plus").to(dev).eval()
paths = [f"{root}/train/{f}" for f in train_df.filename] + [f"{root}/test/{f}" for f in test_df.filename]

def audio_emb(p):
    w, sr = sf.read(p, dtype="float32")
    if w.ndim > 1: w = w.mean(1)
    if sr != 16000: w = resample_poly(w, 16000, sr).astype("float32")
    w = w[:16000*60]
    x = fe(w, sampling_rate=16000, return_tensors="pt").input_values.to(dev)
    with torch.no_grad(): hs = wm(x, output_hidden_states=True).hidden_states
    return torch.cat([hs[i].mean(1).squeeze(0) for i in (4, 8, 12)]).cpu().numpy()

A = np.stack([audio_emb(p) for p in paths])
np.save("/kaggle/working/A.npy", A)
As = StandardScaler().fit(A[:n]).transform(A)
XA = np.hstack([As, Hs])

# CV: audio-only ridge (+ hand features); pick alpha
mask = y > 0
def cv(X, mk):
    o = np.zeros(n)
    for a, b in kf.split(X[:n]):
        m = mk(); m.fit(X[a], y[a]); o[b] = m.predict(X[b])
    return np.clip(o, 0, 5)
def rep(name, p):
    print(name, "| all RMSE %.4f Pearson %.4f | excl.0 RMSE %.4f Pearson %.4f" % (
        np.sqrt(mean_squared_error(y, p)), pearsonr(y, p)[0],
        np.sqrt(mean_squared_error(y[mask], p[mask])), pearsonr(y[mask], p[mask])[0]))
best = None
for al in (300, 1000, 3000, 10000):
    o = cv(XA, lambda: Ridge(alpha=al)); rep(f"ridge_A alpha={al}", o)
    r = np.sqrt(mean_squared_error(y[mask], o[mask]))
    if best is None or r < best[0]: best = (r, al, o)
_, al, oA = best
mA = Ridge(alpha=al).fit(XA[:n], y)
trpA, tepA = np.clip(mA.predict(XA[:n]), 0, 5), np.clip(mA.predict(XA[n:]), 0, 5)

# Blend with the text models
bo2 = (oofs["ridge_H"] + oofs["ridge_EH"] + oofs["svr_EH"] + oA) / 4
bt2 = (trp["ridge_H"] + trp["ridge_EH"] + trp["svr_EH"] + trpA) / 4
bp2 = (tep["ridge_H"] + tep["ridge_EH"] + tep["svr_EH"] + tepA) / 4
rep("BLEND old", bo); rep("BLEND + audio", bo2)
print("TRAIN RMSE blend+audio %.4f" % np.sqrt(mean_squared_error(y, bt2)))
if np.sqrt(mean_squared_error(y[mask], bo2[mask])) < np.sqrt(mean_squared_error(y[mask], bo[mask])):
    pd.DataFrame({"filename": test_df.filename, "label": bp2}).to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated with audio blend")

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

ridge_A alpha=300 | all RMSE 0.5585 Pearson 0.8925 | excl.0 RMSE 0.5711 Pearson 0.8262
ridge_A alpha=1000 | all RMSE 0.5642 Pearson 0.8908 | excl.0 RMSE 0.5773 Pearson 0.8231
ridge_A alpha=3000 | all RMSE 0.5898 Pearson 0.8814 | excl.0 RMSE 0.6035 Pearson 0.8087
ridge_A alpha=10000 | all RMSE 0.6492 Pearson 0.8585 | excl.0 RMSE 0.6643 Pearson 0.7778
BLEND old | all RMSE 0.8533 Pearson 0.7351 | excl.0 RMSE 0.7147 Pearson 0.7154
BLEND + audio | all RMSE 0.7222 Pearson 0.8355 | excl.0 RMSE 0.6327 Pearson 0.7924
TRAIN RMSE blend+audio 0.5262
submission.csv updated with audio blend


In [4]:
r_ = lambda p: np.sqrt(mean_squared_error(y[mask], p[mask]))

# 1. Audio models: lower alphas + SVR
res = {}
for al in (30, 100, 300):
    res[al] = cv(XA, lambda: Ridge(alpha=al)); rep(f"ridge_A alpha={al}", res[al])
oS = cv(XA, lambda: SVR(C=3, epsilon=0.2)); rep("svr_A", oS)
al = min(res, key=lambda a: r_(res[a])); oR = res[al]

mR = Ridge(alpha=al).fit(XA[:n], y); mS = SVR(C=3, epsilon=0.2).fit(XA[:n], y)
c = lambda m, X: np.clip(m.predict(X), 0, 5)
oAud  = (oR + oS) / 2
trAud = (c(mR, XA[:n]) + c(mS, XA[:n])) / 2
teAud = (c(mR, XA[n:]) + c(mS, XA[n:])) / 2
rep("audio ridge+svr", oAud)

# 2. Pick text weight by CV (0 = audio only)
w = min((0, 0.1, 0.2, 0.3, 0.4), key=lambda w: r_((1 - w) * oAud + w * bo))
fo, ft, fp = [(1 - w) * a + w * b for a, b in ((oAud, bo), (trAud, bt), (teAud, bp))]
print("chosen alpha", al, "| text weight", w)
rep("FINAL", fo)
print("TRAIN RMSE %.4f" % np.sqrt(mean_squared_error(y, ft)))

# 3. Write submission only if better than the previous blend
if r_(fo) < r_(bo2):
    sub = pd.DataFrame({"filename": test_df.filename, "label": np.clip(fp, 0, 5)})
    sub.to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated", sub.shape, sub.label.isna().sum())

ridge_A alpha=30 | all RMSE 0.6027 Pearson 0.8745 | excl.0 RMSE 0.6137 Pearson 0.8018
ridge_A alpha=100 | all RMSE 0.5723 Pearson 0.8869 | excl.0 RMSE 0.5842 Pearson 0.8186
ridge_A alpha=300 | all RMSE 0.5585 Pearson 0.8925 | excl.0 RMSE 0.5711 Pearson 0.8262
svr_A | all RMSE 0.5590 Pearson 0.8948 | excl.0 RMSE 0.5645 Pearson 0.8328
audio ridge+svr | all RMSE 0.5448 Pearson 0.8993 | excl.0 RMSE 0.5553 Pearson 0.8378
chosen alpha 300 | text weight 0.2
FINAL | all RMSE 0.5498 Pearson 0.9055 | excl.0 RMSE 0.5466 Pearson 0.8495
TRAIN RMSE 0.2851
submission.csv updated (216, 2) 0


In [5]:
from transformers import WhisperFeatureExtractor, WhisperModel, Wav2Vec2FeatureExtractor

def load16k(p):
    w, sr = sf.read(p, dtype="float32")
    if w.ndim > 1: w = w.mean(1)
    if sr != 16000: w = resample_poly(w, 16000, sr).astype("float32")
    return w[:16000*60]

# --- A. Whisper-small ENCODER embeddings (30 s windows, mean over valid frames) ---
wfe = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
wenc = WhisperModel.from_pretrained("openai/whisper-small").encoder.to(dev).eval()
def whisper_emb(p, layers=(6, 9, 12)):
    w = load16k(p); outs, wts = [], []
    for s in range(0, len(w), 16000*30):
        seg = w[s:s+16000*30]
        if len(seg) < 16000: continue
        x = wfe(seg, sampling_rate=16000, return_tensors="pt").input_features.to(dev)
        with torch.no_grad(): hs = wenc(x, output_hidden_states=True).hidden_states
        nv = max(1, int(len(seg) / 16000 * 50))
        outs.append(torch.cat([hs[i][0, :nv].mean(0) for i in layers]).cpu().numpy()); wts.append(nv)
    if not outs: return np.zeros(768 * len(layers), dtype=np.float32)
    wts = np.array(wts, float) / sum(wts)
    return sum(o * wt for o, wt in zip(outs, wts))
W = np.stack([whisper_emb(p) for p in paths]); np.save("/kaggle/working/W.npy", W)
print("whisper enc", W.shape)

# --- B. HuBERT-base embeddings ---
try: hfe = AutoFeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
except Exception: hfe = Wav2Vec2FeatureExtractor(sampling_rate=16000, do_normalize=True)
hm = AutoModel.from_pretrained("facebook/hubert-base-ls960").to(dev).eval()
def hub_emb(p, layers=(4, 8, 12)):
    x = hfe(load16k(p), sampling_rate=16000, return_tensors="pt").input_values.to(dev)
    with torch.no_grad(): hs = hm(x, output_hidden_states=True).hidden_states
    return torch.cat([hs[i].mean(1).squeeze(0) for i in layers]).cpu().numpy()
Hb = np.stack([hub_emb(p) for p in paths]); np.save("/kaggle/working/Hb.npy", Hb)
print("hubert", Hb.shape)

# --- C. CV each embedding set, then ensemble ---
sets = {"wavlm": XA}
for nm, M in (("whisper_enc", W), ("hubert", Hb)):
    Ms = StandardScaler().fit(M[:n]).transform(M); sets[nm] = np.hstack([Ms, Hs])

def fit_set(X):
    best = None
    for a_ in (100, 300, 1000, 3000):
        o = cv(X, lambda: Ridge(alpha=a_))
        if best is None or r_(o) < best[0]: best = (r_(o), a_, o)
    _, a_, oR = best
    oS = cv(X, lambda: SVR(C=3, epsilon=0.2))
    mR = Ridge(alpha=a_).fit(X[:n], y); mS = SVR(C=3, epsilon=0.2).fit(X[:n], y)
    return (oR + oS) / 2, (c(mR, X[:n]) + c(mS, X[:n])) / 2, (c(mR, X[n:]) + c(mS, X[n:])) / 2, a_

R = {}
for nm, X in sets.items():
    R[nm] = fit_set(X); rep(f"{nm} (alpha={R[nm][3]})", R[nm][0])

oE, tE, pE = [np.mean([R[k][i] for k in R], 0) for i in range(3)]
rep("audio ensemble", oE)
w = min((0, 0.1, 0.2, 0.3), key=lambda w: r_((1 - w) * oE + w * bo))
fo2, ft2, fp2 = [(1 - w) * a + w * b for a, b in ((oE, bo), (tE, bt), (pE, bp))]
print("text weight", w); rep("FINAL v2", fo2); rep("FINAL v1 (submitted)", fo)
print("TRAIN RMSE v2 %.4f" % np.sqrt(mean_squared_error(y, ft2)))
if r_(fo2) < r_(fo):
    sub = pd.DataFrame({"filename": test_df.filename, "label": np.clip(fp2, 0, 5)})
    sub.to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated", sub.shape, sub.label.isna().sum())

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

hubert (985, 2304)
wavlm (alpha=300) | all RMSE 0.5448 Pearson 0.8993 | excl.0 RMSE 0.5553 Pearson 0.8378
whisper_enc (alpha=1000) | all RMSE 0.5388 Pearson 0.9041 | excl.0 RMSE 0.5332 Pearson 0.8525
hubert (alpha=300) | all RMSE 0.5504 Pearson 0.8981 | excl.0 RMSE 0.5586 Pearson 0.8364
audio ensemble | all RMSE 0.5216 Pearson 0.9109 | excl.0 RMSE 0.5272 Pearson 0.8581
text weight 0.1
FINAL v2 | all RMSE 0.5262 Pearson 0.9136 | excl.0 RMSE 0.5249 Pearson 0.8629
FINAL v1 (submitted) | all RMSE 0.5498 Pearson 0.9055 | excl.0 RMSE 0.5466 Pearson 0.8495
TRAIN RMSE v2 0.2516
submission.csv updated (216, 2) 0


In [6]:
from scipy.stats import rankdata
import gc

def emb_set(model, fe_fn, layers, fname):
    outs = []
    for p in paths:
        x = fe_fn(load16k(p)).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            hs = model(x, output_hidden_states=True).hidden_states
        outs.append(torch.cat([hs[i].float().mean(1).squeeze(0) for i in layers]).cpu().numpy())
    M = np.stack(outs); np.save(f"/kaggle/working/{fname}.npy", M); return M

# --- A. WavLM-large (layers spread across depth) ---
wl_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-large")
wl = AutoModel.from_pretrained("microsoft/wavlm-large").to(dev).eval()
WL = emb_set(wl, lambda w: wl_fe(w, sampling_rate=16000, return_tensors="pt").input_values,
             (6, 12, 18, 24), "WL")
print("wavlm-large", WL.shape); del wl; gc.collect(); torch.cuda.empty_cache()

# --- B. Whisper-medium encoder (30 s windows, mean over valid frames) ---
wm_fe = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")
wm_enc = WhisperModel.from_pretrained("openai/whisper-medium").encoder.to(dev).eval()
def whisper_med(p, layers=(8, 16, 24)):
    w = load16k(p); outs, wts = [], []
    for s in range(0, len(w), 16000*30):
        seg = w[s:s+16000*30]
        if len(seg) < 16000: continue
        x = wm_fe(seg, sampling_rate=16000, return_tensors="pt").input_features.to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            hs = wm_enc(x, output_hidden_states=True).hidden_states
        nv = max(1, int(len(seg) / 16000 * 50))
        outs.append(torch.cat([hs[i][0, :nv].float().mean(0) for i in layers]).cpu().numpy()); wts.append(nv)
    if not outs: return np.zeros(1024 * len(layers), dtype=np.float32)
    wts = np.array(wts, float) / sum(wts)
    return sum(o * wt for o, wt in zip(outs, wts))
WM = np.stack([whisper_med(p) for p in paths]); np.save("/kaggle/working/WM.npy", WM)
print("whisper-medium enc", WM.shape); del wm_enc; gc.collect(); torch.cuda.empty_cache()

# --- C. CV each new set ---
for nm, M in (("wavlm_large", WL), ("whisper_med", WM)):
    Ms = StandardScaler().fit(M[:n]).transform(M)
    R[nm] = fit_set(np.hstack([Ms, Hs])); rep(f"{nm} (alpha={R[nm][3]})", R[nm][0])

# --- D. Stack: non-negative Ridge on out-of-fold predictions (+ text blend) ---
names = list(R)
S_oof = np.column_stack([R[k][0] for k in names] + [bo])
S_tr  = np.column_stack([R[k][1] for k in names] + [bt])
S_te  = np.column_stack([R[k][2] for k in names] + [bp])
kf2 = KFold(5, shuffle=True, random_state=7)          # different folds for the stacker
o_st = np.zeros(n)
for a, b in kf2.split(S_oof):
    st_ = Ridge(alpha=1.0, positive=True).fit(S_oof[a], y[a]); o_st[b] = st_.predict(S_oof[b])
o_st = np.clip(o_st, 0, 5)
stk = Ridge(alpha=1.0, positive=True).fit(S_oof, y)
print("stack weights:", dict(zip(names + ["text_blend"], stk.coef_.round(3))), "intercept %.3f" % stk.intercept_)

rep("simple mean of all audio", np.mean([R[k][0] for k in names], 0))
rep("STACKED v3", o_st); rep("FINAL v2 (current best)", fo2)
ft3, fp3 = np.clip(stk.predict(S_tr), 0, 5), np.clip(stk.predict(S_te), 0, 5)
print("TRAIN RMSE v3 %.4f" % np.sqrt(mean_squared_error(y, ft3)))
if r_(o_st) < r_(fo2) - 0.003:        # require a clear gain, not noise
    sub = pd.DataFrame({"filename": test_df.filename, "label": fp3})
    sub.to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated", sub.shape, sub.label.isna().sum())
else:
    print("no clear gain: submission.csv unchanged")

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.06GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

whisper-medium enc (985, 3072)
wavlm_large (alpha=1000) | all RMSE 0.5224 Pearson 0.9089 | excl.0 RMSE 0.5291 Pearson 0.8547
whisper_med (alpha=1000) | all RMSE 0.5194 Pearson 0.9113 | excl.0 RMSE 0.5156 Pearson 0.8632
stack weights: {'wavlm': np.float64(0.0), 'whisper_enc': np.float64(0.002), 'hubert': np.float64(0.258), 'wavlm_large': np.float64(0.322), 'whisper_med': np.float64(0.443), 'text_blend': np.float64(0.146)} intercept -0.566
simple mean of all audio | all RMSE 0.5121 Pearson 0.9147 | excl.0 RMSE 0.5164 Pearson 0.8646
STACKED v3 | all RMSE 0.4920 Pearson 0.9178 | excl.0 RMSE 0.4992 Pearson 0.8705
FINAL v2 (current best) | all RMSE 0.5262 Pearson 0.9136 | excl.0 RMSE 0.5249 Pearson 0.8629
TRAIN RMSE v3 0.2020
submission.csv updated (216, 2) 0


In [7]:
import gc
from transformers import WhisperFeatureExtractor, WhisperModel

# 1. Whisper-large-v3 encoder (fp16), mean over valid frames of 30 s windows
l_fe = WhisperFeatureExtractor.from_pretrained("openai/whisper-large-v3")
l_enc = WhisperModel.from_pretrained("openai/whisper-large-v3", torch_dtype=torch.float16).encoder.to(dev).eval()
def whisper_l3(p, layers=(16, 24, 32)):
    w = load16k(p); outs, wts = [], []
    for s in range(0, len(w), 16000*30):
        seg = w[s:s+16000*30]
        if len(seg) < 16000: continue
        x = l_fe(seg, sampling_rate=16000, return_tensors="pt").input_features.to(dev).half()
        with torch.no_grad(): hs = l_enc(x, output_hidden_states=True).hidden_states
        nv = max(1, int(len(seg) / 16000 * 50))
        outs.append(torch.cat([hs[i][0, :nv].float().mean(0) for i in layers]).cpu().numpy()); wts.append(nv)
    if not outs: return np.zeros(1280 * len(layers), dtype=np.float32)
    wts = np.array(wts, float) / sum(wts)
    return sum(o * wt for o, wt in zip(outs, wts))
WL3 = np.stack([whisper_l3(p) for p in paths]); np.save("/kaggle/working/WL3.npy", WL3)
print("whisper-large-v3", WL3.shape); del l_enc; gc.collect(); torch.cuda.empty_cache()

# 2. Stacker averaged over 3 CV seeds (more stable than one split)
def stack(names):
    So = np.column_stack([R[k][0] for k in names] + [bo])
    St = np.column_stack([R[k][1] for k in names] + [bt])
    Sp = np.column_stack([R[k][2] for k in names] + [bp])
    oo = np.zeros(n)
    for sd in (7, 11, 13):
        for a, b in KFold(5, shuffle=True, random_state=sd).split(So):
            oo[b] += Ridge(alpha=1.0, positive=True).fit(So[a], y[a]).predict(So[b]) / 3
    m = Ridge(alpha=1.0, positive=True).fit(So, y)
    return np.clip(oo, 0, 5), np.clip(m.predict(St), 0, 5), np.clip(m.predict(Sp), 0, 5), m

old_names = list(R)
o_old, t_old, p_old, _ = stack(old_names)

Ms = StandardScaler().fit(WL3[:n]).transform(WL3)
R["whisper_l3"] = fit_set(np.hstack([Ms, Hs])); rep(f"whisper_l3 (alpha={R['whisper_l3'][3]})", R["whisper_l3"][0])
o_new, t_new, p_new, m_new = stack(list(R))

print("weights:", dict(zip(list(R) + ["text_blend"], m_new.coef_.round(3))))
rep("STACK old (3-seed)", o_old); rep("STACK + large-v3 (3-seed)", o_new)
print("TRAIN RMSE new %.4f" % np.sqrt(mean_squared_error(y, t_new)))
if r_(o_new) < r_(o_old) - 0.003:
    pd.DataFrame({"filename": test_df.filename, "label": p_new}).to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated")
else:
    print("no clear gain: submission.csv unchanged")

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

whisper-large-v3 (985, 3840)
whisper_l3 (alpha=1000) | all RMSE 0.5280 Pearson 0.9074 | excl.0 RMSE 0.5207 Pearson 0.8594
weights: {'wavlm': np.float64(0.0), 'whisper_enc': np.float64(0.0), 'hubert': np.float64(0.254), 'wavlm_large': np.float64(0.289), 'whisper_med': np.float64(0.328), 'whisper_l3': np.float64(0.154), 'text_blend': np.float64(0.149)}
STACK old (3-seed) | all RMSE 0.4918 Pearson 0.9179 | excl.0 RMSE 0.4990 Pearson 0.8706
STACK + large-v3 (3-seed) | all RMSE 0.4914 Pearson 0.9180 | excl.0 RMSE 0.4981 Pearson 0.8711
TRAIN RMSE new 0.2026
no clear gain: submission.csv unchanged


In [8]:
import itertools
def tune_svr(fname):
    M = np.load(f"/kaggle/working/{fname}.npy"); Ms = StandardScaler().fit(M[:n]).transform(M)
    X = np.hstack([Ms, Hs]); out = []
    for C, eps in itertools.product((1, 3, 10), (0.1, 0.2, 0.4)):
        o = cv(X, lambda: SVR(C=C, epsilon=eps)); out.append((r_(o), C, eps))
    return sorted(out)[:3]
for nm, f in (("whisper_med", "WM"), ("wavlm_large", "WL"), ("hubert", "Hb")):
    print(nm, [(round(r, 4), C, e) for r, C, e in tune_svr(f)])

whisper_med [(np.float64(0.5082), 10, 0.1), (np.float64(0.5082), 3, 0.1), (np.float64(0.5175), 10, 0.2)]
wavlm_large [(np.float64(0.5239), 10, 0.1), (np.float64(0.5265), 3, 0.1), (np.float64(0.5342), 10, 0.2)]
hubert [(np.float64(0.5625), 10, 0.1), (np.float64(0.5649), 3, 0.1), (np.float64(0.5715), 10, 0.2)]


In [9]:
R_old = {k: v for k, v in R.items() if k != "whisper_l3"}      # current setup (eps=0.2)

def fit_set2(X, C=3, eps=0.1):
    best = None
    for a_ in (100, 300, 1000, 3000):
        o = cv(X, lambda: Ridge(alpha=a_))
        if best is None or r_(o) < best[0]: best = (r_(o), a_, o)
    _, a_, oR = best
    oS = cv(X, lambda: SVR(C=C, epsilon=eps))
    mR = Ridge(alpha=a_).fit(X[:n], y); mS = SVR(C=C, epsilon=eps).fit(X[:n], y)
    return (oR + oS) / 2, (c(mR, X[:n]) + c(mS, X[:n])) / 2, (c(mR, X[n:]) + c(mS, X[n:])) / 2, a_

R_new = {}
for nm, f in (("wavlm", "A"), ("whisper_enc", "W"), ("hubert", "Hb"), ("wavlm_large", "WL"), ("whisper_med", "WM")):
    M = np.load(f"/kaggle/working/{f}.npy"); Ms = StandardScaler().fit(M[:n]).transform(M)
    R_new[nm] = fit_set2(np.hstack([Ms, Hs])); rep(f"{nm} eps=0.1 (alpha={R_new[nm][3]})", R_new[nm][0])

R = R_old; o_old, t_old, p_old, _ = stack(list(R))
R = R_new; o_new, t_new, p_new, m_new = stack(list(R))
print("weights:", dict(zip(list(R) + ["text_blend"], m_new.coef_.round(3))))
rep("STACK eps=0.2 (current)", o_old); rep("STACK eps=0.1", o_new)
print("TRAIN RMSE new %.4f" % np.sqrt(mean_squared_error(y, t_new)))
if r_(o_new) < r_(o_old) - 0.003:
    pd.DataFrame({"filename": test_df.filename, "label": p_new}).to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated")
else:
    print("no clear gain: keep current setup")

wavlm eps=0.1 (alpha=300) | all RMSE 0.5409 Pearson 0.9006 | excl.0 RMSE 0.5517 Pearson 0.8398
whisper_enc eps=0.1 (alpha=1000) | all RMSE 0.5344 Pearson 0.9050 | excl.0 RMSE 0.5297 Pearson 0.8541
hubert eps=0.1 (alpha=300) | all RMSE 0.5468 Pearson 0.8989 | excl.0 RMSE 0.5555 Pearson 0.8378
wavlm_large eps=0.1 (alpha=1000) | all RMSE 0.5172 Pearson 0.9102 | excl.0 RMSE 0.5244 Pearson 0.8569
whisper_med eps=0.1 (alpha=1000) | all RMSE 0.5147 Pearson 0.9123 | excl.0 RMSE 0.5117 Pearson 0.8649
weights: {'wavlm': np.float64(0.0), 'whisper_enc': np.float64(0.0), 'hubert': np.float64(0.244), 'wavlm_large': np.float64(0.329), 'whisper_med': np.float64(0.439), 'text_blend': np.float64(0.147)}
STACK eps=0.2 (current) | all RMSE 0.4918 Pearson 0.9179 | excl.0 RMSE 0.4990 Pearson 0.8706
STACK eps=0.1 | all RMSE 0.4892 Pearson 0.9188 | excl.0 RMSE 0.4964 Pearson 0.8720
TRAIN RMSE new 0.1767
no clear gain: keep current setup


In [10]:
# A. Fluency features from the raw audio: pauses, speech ratio, articulation rate
def vad_feats(p, nwords):
    w, sr = sf.read(p, dtype="float32")
    if w.ndim > 1: w = w.mean(1)
    fr, hop = int(0.025 * sr), int(0.010 * sr)
    win = np.lib.stride_tricks.sliding_window_view(w, fr)[::hop]
    db = 10 * np.log10((win ** 2).mean(1) + 1e-10)
    sp = db > (np.percentile(db, 95) - 30)
    if sp.sum() < 5: return [len(w) / sr] + [0.0] * 10
    idx = np.where(sp)[0]; t = sp[idx[0]: idx[-1] + 1]
    dur = len(t) * 0.01
    edges = np.diff(np.r_[0, (~t).astype(int), 0])
    runs = (np.where(edges == -1)[0] - np.where(edges == 1)[0]) * 0.01
    pz = runs[runs > 0.25]
    speech_t = max(t.sum() * 0.01, 1e-3)
    return [len(w) / sr, t.mean(), len(pz), (pz > 0.7).sum(), pz.mean() if len(pz) else 0.0,
            pz.max() if len(pz) else 0.0, len(pz) / dur, nwords / speech_t, nwords / dur,
            db.std(), np.percentile(db, 95) - np.percentile(db, 10)]

nw = np.concatenate([Xtr.n_words.values, Xte.n_words.values])
V = np.array([vad_feats(p, k) for p, k in zip(paths, nw)]); np.save("/kaggle/working/V.npy", V)
Vs = StandardScaler().fit(V[:n]).transform(V)

def fit_low(X):                      # low-dimensional features need small alphas
    best = None
    for a_ in (1, 10, 30, 100, 300):
        o = cv(X, lambda: Ridge(alpha=a_))
        if best is None or r_(o) < best[0]: best = (r_(o), a_, o)
    _, a_, oR = best
    oS = cv(X, lambda: SVR(C=3, epsilon=0.1))
    mR = Ridge(alpha=a_).fit(X[:n], y); mS = SVR(C=3, epsilon=0.1).fit(X[:n], y)
    return (oR + oS) / 2, (c(mR, X[:n]) + c(mS, X[:n])) / 2, (c(mR, X[n:]) + c(mS, X[n:])) / 2, a_

old_names = list(R); o_old, t_old, p_old, _ = stack(old_names)
R["fluency"] = fit_low(np.hstack([Hs, Vs])); rep(f"fluency feats (alpha={R['fluency'][3]})", R["fluency"][0])
o_new, t_new, p_new, m_new = stack(list(R))
print("weights:", dict(zip(list(R) + ["text_blend"], m_new.coef_.round(3))))
rep("STACK old", o_old); rep("STACK + fluency", o_new)
print("TRAIN RMSE new %.4f" % np.sqrt(mean_squared_error(y, t_new)))
if r_(o_new) < r_(o_old) - 0.003:
    pd.DataFrame({"filename": test_df.filename, "label": p_new}).to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated")
else:
    print("no clear gain: submission.csv unchanged")

# B. Diagnostic only: do the 37 garbled label-0 files hurt the fit for the other files?
M = np.load("/kaggle/working/WM.npy"); Xm = np.hstack([StandardScaler().fit(M[:n]).transform(M), Hs])
for wz in (1.0, 0.3, 0.0):
    sw = np.where(y > 0, 1.0, wz); o = np.zeros(n)
    for a, b in kf.split(Xm[:n]):
        r1 = Ridge(alpha=1000).fit(Xm[a], y[a], sample_weight=sw[a])
        s1 = SVR(C=3, epsilon=0.1).fit(Xm[a], y[a], sample_weight=sw[a])
        o[b] = (r1.predict(Xm[b]) + s1.predict(Xm[b])) / 2
    print("zero-weight %.1f -> excl.0 RMSE %.4f" % (wz, r_(np.clip(o, 0, 5))))

fluency feats (alpha=30) | all RMSE 0.7548 Pearson 0.7971 | excl.0 RMSE 0.7229 Pearson 0.7032
weights: {'wavlm': np.float64(0.018), 'whisper_enc': np.float64(0.047), 'hubert': np.float64(0.235), 'wavlm_large': np.float64(0.316), 'whisper_med': np.float64(0.404), 'fluency': np.float64(0.031), 'text_blend': np.float64(0.117)}
STACK old | all RMSE 0.4892 Pearson 0.9188 | excl.0 RMSE 0.4964 Pearson 0.8720
STACK + fluency | all RMSE 0.4896 Pearson 0.9187 | excl.0 RMSE 0.4968 Pearson 0.8718
TRAIN RMSE new 0.1780
no clear gain: submission.csv unchanged
zero-weight 1.0 -> excl.0 RMSE 0.5117
zero-weight 0.3 -> excl.0 RMSE 0.5106
zero-weight 0.0 -> excl.0 RMSE 0.5100


In [11]:
import gc
from transformers import AutoFeatureExtractor, AutoModel, WhisperFeatureExtractor, WhisperModel

def pool(frames):   # list of (T, D) tensors, one per layer -> mean + std per layer
    return torch.cat([torch.cat([f.mean(0), f.std(0)]) for f in frames]).cpu().numpy()

# A. WavLM-large: mean+std over 7 layers
fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-large")
wl = AutoModel.from_pretrained("microsoft/wavlm-large").to(dev).eval()
L1 = (6, 9, 12, 15, 18, 21, 24)
def wl_ms(p):
    x = fe(load16k(p), sampling_rate=16000, return_tensors="pt").input_values.to(dev)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        hs = wl(x, output_hidden_states=True).hidden_states
    return pool([hs[i][0].float() for i in L1])
WLms = np.stack([wl_ms(p) for p in paths]); np.save("/kaggle/working/WLms.npy", WLms)
print("wavlm-large ms", WLms.shape); del wl; gc.collect(); torch.cuda.empty_cache()

# B. Whisper-medium encoder: mean+std over 5 layers, pooled over all valid frames
wfe = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")
wenc = WhisperModel.from_pretrained("openai/whisper-medium").encoder.to(dev).eval()
L2 = (8, 12, 16, 20, 24)
def wm_ms(p):
    w = load16k(p); fr = [[] for _ in L2]
    for s in range(0, len(w), 16000 * 30):
        seg = w[s:s + 16000 * 30]
        if len(seg) < 16000: continue
        x = wfe(seg, sampling_rate=16000, return_tensors="pt").input_features.to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            hs = wenc(x, output_hidden_states=True).hidden_states
        nv = max(2, int(len(seg) / 16000 * 50))
        for j, i in enumerate(L2): fr[j].append(hs[i][0, :nv].float())
    if not fr[0]: return np.zeros(1024 * 2 * len(L2), dtype=np.float32)
    return pool([torch.cat(f) for f in fr])
WMms = np.stack([wm_ms(p) for p in paths]); np.save("/kaggle/working/WMms.npy", WMms)
print("whisper-medium ms", WMms.shape); del wenc; gc.collect(); torch.cuda.empty_cache()

# C. CV the new sets and re-stack
base = ["wavlm", "whisper_enc", "hubert", "wavlm_large", "whisper_med"]
fit_s = fit_set2 if "fit_set2" in globals() else fit_set
for nm, M in (("wavlm_large_ms", WLms), ("whisper_med_ms", WMms)):
    Ms = StandardScaler().fit(M[:n]).transform(M)
    R[nm] = fit_s(np.hstack([Ms, Hs])); rep(f"{nm} (alpha={R[nm][3]})", R[nm][0])

o_old, t_old, p_old, _ = stack(base)
o_new, t_new, p_new, m_new = stack(base + ["wavlm_large_ms", "whisper_med_ms"])
print("weights:", dict(zip(base + ["wavlm_large_ms", "whisper_med_ms", "text_blend"], m_new.coef_.round(3))))
rep("STACK base", o_old); rep("STACK + ms-pooled", o_new)
print("TRAIN RMSE new %.4f" % np.sqrt(mean_squared_error(y, t_new)))
if r_(o_new) < r_(o_old) - 0.003:
    pd.DataFrame({"filename": test_df.filename, "label": p_new}).to_csv("/kaggle/working/submission.csv", index=False)
    print("submission.csv updated")
else:
    print("no clear gain: submission.csv unchanged")

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

wavlm-large ms (985, 14336)


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

whisper-medium ms (985, 10240)
wavlm_large_ms (alpha=1000) | all RMSE 0.5135 Pearson 0.9111 | excl.0 RMSE 0.5219 Pearson 0.8580
whisper_med_ms (alpha=1000) | all RMSE 0.5099 Pearson 0.9137 | excl.0 RMSE 0.5058 Pearson 0.8679
weights: {'wavlm': np.float64(0.0), 'whisper_enc': np.float64(0.0), 'hubert': np.float64(0.206), 'wavlm_large': np.float64(0.135), 'whisper_med': np.float64(0.164), 'wavlm_large_ms': np.float64(0.189), 'whisper_med_ms': np.float64(0.322), 'text_blend': np.float64(0.137)}
STACK base | all RMSE 0.4892 Pearson 0.9188 | excl.0 RMSE 0.4964 Pearson 0.8720
STACK + ms-pooled | all RMSE 0.4849 Pearson 0.9202 | excl.0 RMSE 0.4915 Pearson 0.8747
TRAIN RMSE new 0.1488
submission.csv updated
